# tinyLMTune — Text Classification

This notebook demonstrates 3 ways to train TinyBERT for **classification** using tinyLMTune:

1. **Synthetic data** — auto-generated via Flan-T5/Mistral
2. **Benchmark data** — real HuggingFace dataset (rotten_tomatoes)
3. **Raw user data** — your own text, structured or unstructured

Each example runs the full pipeline: data → token analysis → search space recommendation → GA optimisation → model save → inference.

## Setup

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
pip install tinylmtune

In [ ]:
from tinylmtune import optimize_slm, TinyInference, print_token_analysis, print_recommendation, plot_results

In [ ]:
## Check search space recommendation before you start training 

print_recommendation(n_samples=200, task="classification")

---
## Example 1 — Synthetic Data (via Flan-T5)

No data needed. Flan-T5/Mistral generates training data from a topic prompt.

**Requirements:** Flan-T5 must be installed and running (`pip install sentencepiece`), with `mistral` model pulled (``).

In [ ]:
best = optimize_slm(
    task="classification",
    corpus_prompt="Generate diverse product review sentiment examples",
    n_examples=200,
    labels="positive,negative,neutral",
    pop_size=4,
    generations=5,
    output_dir="models/classification_synthetic",
)
#print("Best config:", best)



In [ ]:
# Only displays, doesn't save
plot_results(best, save_dir="plots/synthetic_data")

### Inference on synthetic model

In [ ]:
model = TinyInference("models/classification_synthetic")

for text in [
    "Absolutely love this product, best purchase ever!",
    "Terrible quality, broke after one day.",
    "It's fine, does what it says.",
]:
    result = model.predict(text)
    print(f"Text:  {text}")
    print(f"Label: {result['label']}  Confidence: {result['confidence']:.1%}")
    print()


---
## Example 2 — Benchmark Data (rotten_tomatoes)

Uses a real HuggingFace dataset. No Flan-T5 needed.

### Load rotten_tomatoes dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("rotten_tomatoes", split="train")
ds = ds.shuffle(seed=42).select(range(1000))  # use 300 samples

label_map = {0: "negative", 1: "positive"}
benchmark_data = [{"text": r["text"], "label": label_map[r["label"]]} for r in ds]

print(f"Loaded {len(benchmark_data)} records")
print(f"Sample: {benchmark_data[0]}")

### Analyze token lengths

In [ ]:
print_token_analysis(benchmark_data, task="classification")

### Check recommended search space

In [ ]:
print_recommendation(n_samples=len(benchmark_data), task="classification")

### Train with GA optimisation

In [ ]:
best = optimize_slm(
    task="classification",
    user_data=benchmark_data,
    pop_size=6,
    generations=3,
    max_len = 64,
    output_dir="models/classification_benchmark",
)
print("Best config:", best)

### Visualize GA Results

5 plots showing how the GA searched for the best hyperparameters:
1. **Fitness progress** — best/avg/worst per generation
2. **Parameter scatter** — each param vs fitness (best = red star)
3. **Scheduler comparison** — box plot by LR scheduler type
4. **Config evolution** — how the best config changed over generations
5. **Population heatmap** — all individuals in the last generation

In [ ]:
from tinylmtune import plot_results, print_best_config_table

# Print formatted best config
print_best_config_table(best)

# Generate all 5 plots
figs = plot_results(best, save_dir="plots/benchmark")

### Inference on benchmark model

In [ ]:
model = TinyInference("models/classification_benchmark")

test_texts = [
    "A stunning visual masterpiece with brilliant performances.",
    "Predictable plot and terrible dialogue throughout.",
    "An average film that neither excels nor disappoints.",
]
for text in test_texts:
    result = model.predict(text)
    print(f"{result['label']:10s} ({result['confidence']:.1%})  {text}")

---
## Example 3 — Raw User Data

Three sub-examples showing different input formats:
- **3a.** Structured dicts (correct format)
- **3b.** Raw text strings (auto-labelled via Flan-T5)
- **3c.** Wrong-format dicts (auto-detected and converted)

### 3a. Structured dicts (used directly, no Flan-T5)

In [ ]:
my_data = [
    {"text": "Absolutely loved this product!", "label": "positive"},
    {"text": "Best purchase I've made this year", "label": "positive"},
    {"text": "Works great, highly recommend", "label": "positive"},
    {"text": "Exceeded all my expectations", "label": "positive"},
    {"text": "Amazing quality for the price", "label": "positive"},
    {"text": "Will definitely buy again", "label": "positive"},
    {"text": "Perfect gift, they loved it", "label": "positive"},
    {"text": "Five stars, no complaints at all", "label": "positive"},
    {"text": "Total waste of money", "label": "negative"},
    {"text": "Broke after just two days", "label": "negative"},
    {"text": "Worst product I've ever bought", "label": "negative"},
    {"text": "Completely unusable, returning it", "label": "negative"},
    {"text": "Poor quality, very disappointed", "label": "negative"},
    {"text": "Do not buy, save your money", "label": "negative"},
    {"text": "Terrible customer service too", "label": "negative"},
    {"text": "Fell apart on first use", "label": "negative"},
    {"text": "It's okay, nothing special", "label": "neutral"},
    {"text": "Average product, does the job", "label": "neutral"},
    {"text": "Meets expectations, no more no less", "label": "neutral"},
    {"text": "Fine for the price, not amazing", "label": "neutral"},
]

best = optimize_slm(
    task="classification",
    user_data=my_data,
    pop_size=4,
    generations=2,
    output_dir="models/classification_user",
)

### 3b. Raw text strings (requires Flan-T5)

In [ ]:
# Raw strings — Flan-T5 will label them automatically
raw_texts = [
    "This movie was absolutely wonderful and heartwarming!",
    "A complete waste of two hours of my life.",
    "The acting was superb but the plot dragged on.",
    "One of the best films I've seen this year!",
    "Terribly written with no character development.",
    "Decent entertainment, nothing groundbreaking.",
]

labels = "positive,negative,neutral"

best = optimize_slm(
    task="classification",
    user_data=raw_texts,
    labels= labels,
    pop_size=4,
    generations=1,
    output_dir="models/classification_raw",
)

### 3c. Wrong-format dicts (requires Flan-T5)

In [ ]:
# Dicts with wrong keys — pipeline auto-detects and converts
wrong_format = [
    {"content": "This restaurant has amazing food and great service!"},
    {"content": "Worst dining experience ever, never going back."},
    {"content": "The food was okay but overpriced for the quality."},
    {"content": "Absolutely delicious, my new favourite place!"},
    {"content": "Rude staff and cold food, very disappointed."},
    {"content": "Average meal, nothing to write home about."},
]

# Pipeline detects "content" key, extracts text, labels via Flan-T5
best = optimize_slm(
    task="classification",
    user_data=wrong_format,
    labels="positive,negative,neutral",
    pop_size=4,
    generations=1,
    output_dir="models/classification_wrong",
)

### Visualize user data results

In [ ]:
# Plot results from structured data training (Example 3a)
from tinylmtune import plot_results, print_best_config_table
print_best_config_table(best)
figs = plot_results(best, save_dir="plots/user_data")

### Inference

In [ ]:
model = TinyInference("models/classification_user")
result = model.predict("This is the best thing I've ever purchased!")
print(result)

---
## Summary

| Example | Data source | Flan-T5 needed | Best for |
|---------|-------------|---------------|----------|
| Synthetic | Auto-generated | Yes | Quick prototyping |
| Benchmark | rotten_tomatoes | No | Reproducible evaluation |
| User data | Your own text | Depends on format | Production use |

The GA searches 11 hyperparameters: `learning_rate`, `batch_size`, `epochs`, `warmup_ratio`, `weight_decay`, `dropout`, `attention_dropout`, `gradient_accumulation_steps`, `lr_scheduler_type`, `label_smoothing`, `max_grad_norm`.

`max_len` is automatically determined from your data's token length distribution (p95 percentile).